# VLM-SAM3 Results — Jobs 465388 & 465389

**Job 465388** — EXP-C-box / EXP-C-box-cot (bbox localization approach)  
**Job 465389** — EXP-F-mf1 / mf3 / mf5 / mf10 (multiframe ablation)  
**Completed**: 2026-03-09

### What's new in these jobs
- **EXP-C-box**: VLM outputs a bounding box (x1,y1,x2,y2) instead of text. SAM3 uses it as a geometry prompt — bypasses the 32-token text encoder limit entirely.
- **EXP-F-mf{k}**: Ablation of how many source frames to show the VLM (1, 3, 5, 10). All use instruct model.
- **Critical fix**: Qwen3-VL outputs bounding boxes in 0-1000 normalized space. Previous jobs forgot to divide by 1000 → IoU=0 for all bbox pairs. Fixed in commit `ece420a`.

### Sections
1. Setup & data loading
2. Overall summary table (all experiments)
3. EXP-C-box deep dive: bbox parse rate, IoU distribution
4. EXP-C-box vs text baselines
5. Per-scenario breakdown (EXP-C-box)
6. Multiframe ablation (EXP-F)
7. Best/worst examples
8. Key findings

In [ ]:
from pathlib import Path

REPO_ROOT  = Path("/home/3164542/reluminati-research")
SCRIPT_DIR = REPO_ROOT / "src/scripts/experiments/vlm-sam3"

# All result files (deduplicated below)
RESULT_FILES = [
    SCRIPT_DIR / "results_463119.csv",
    SCRIPT_DIR / "results_465254.csv",
    SCRIPT_DIR / "results_465301.csv",
    SCRIPT_DIR / "results_465302.csv",
    SCRIPT_DIR / "results_465303.csv",
    SCRIPT_DIR / "results_465388.csv",   # EXP-C-box/cot  ← NEW
    SCRIPT_DIR / "results_465389.csv",   # EXP-F-mf1/3/5/10 ← NEW
]

BASELINE_CSV = Path("/data/video_datasets/3164542/baseline_multiframe_w10.csv")

# Experiments in each new job
BOX_EXPS = ["EXP-C-box", "EXP-C-box-cot"]
MF_EXPS  = ["EXP-F-mf1", "EXP-F-mf3", "EXP-F-mf5", "EXP-F-mf10"]
TEXT_BASELINES = ["EXP-C", "EXP-C-cot"]
ALL_TARGET_EXPS = TEXT_BASELINES + BOX_EXPS + MF_EXPS

LM_EEC_MEAN_IOU = 0.660   # exo→ego direction

print(f"Script dir  : {SCRIPT_DIR}")
print(f"Baseline    : {BASELINE_CSV}  exists={BASELINE_CSV.exists()}")

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 5), "font.size": 11})
sns.set_theme(style="whitegrid", palette="husl")

EXP_PALETTE = {
    "EXP-C":          "#3498db",
    "EXP-C-cot":      "#e74c3c",
    "EXP-C-box":      "#f39c12",
    "EXP-C-box-cot":  "#e67e22",
    "EXP-F-mf1":      "#27ae60",
    "EXP-F-mf3":      "#16a085",
    "EXP-F-mf5":      "#2980b9",
    "EXP-F-mf10":     "#8e44ad",
}

EXP_LABELS = {
    "EXP-C":          "EXP-C\n(cloud,text)",
    "EXP-C-cot":      "EXP-C-cot\n(instruct,text)",
    "EXP-C-box":      "EXP-C-box\n(cloud,bbox)",
    "EXP-C-box-cot":  "EXP-C-box-cot\n(instruct,bbox)",
    "EXP-F-mf1":      "mf1\n(1 frame)",
    "EXP-F-mf3":      "mf3\n(3 frames)",
    "EXP-F-mf5":      "mf5\n(5 frames)",
    "EXP-F-mf10":     "mf10\n(10 frames)",
}

KEY_COLS = ["take_uid", "object_name", "src_camera", "dest_camera", "frame", "experiment"]
print("Imports OK.")

---
## 1  Load & Deduplicate Data

In [ ]:
dfs = []
for f in RESULT_FILES:
    if f.exists():
        dfs.append(pd.read_csv(f, dtype=str))
        print(f"Loaded: {f.name}  ({len(dfs[-1])} rows)")
    else:
        print(f"SKIP (not found): {f.name}")

raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal before dedup : {len(raw):,}")

# Deduplicate: keep last (most recent job wins)
all_df = raw.drop_duplicates(subset=KEY_COLS, keep="last").copy()
print(f"Total after dedup  : {len(all_df):,}")

# Type coercion
all_df["iou"] = pd.to_numeric(all_df["iou"], errors="coerce")
all_df["success"] = all_df["iou"] > 0

print(f"\nExperiments present:")
print(all_df["experiment"].value_counts().to_string())

---
## 2  Overall Summary Table

In [ ]:
def exp_stats(df, exp):
    g = df[df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    n = len(iou)
    return {
        "N":           n,
        "iou>0 %":     100 * len(pos) / max(1, n),
        "mean_all":    iou.mean(),
        "mean_pos":    pos.mean() if len(pos) else float("nan"),
        "iou>0.5 %":  100 * (pos > 0.5).mean() if len(pos) else 0,
        "iou>0.75 %": 100 * (pos > 0.75).mean() if len(pos) else 0,
    }

rows = []
for exp in ALL_TARGET_EXPS:
    s = exp_stats(all_df, exp)
    s["experiment"] = exp
    rows.append(s)

summ = pd.DataFrame(rows).set_index("experiment")

# Add LM-EEC baseline row for reference
lm_row = pd.DataFrame([{
    "N": "—", "iou>0 %": "—", "mean_all": 0.660,
    "mean_pos": 0.660, "iou>0.5 %": "—", "iou>0.75 %": "—"
}], index=["LM-EEC (exo→ego)"])

display_df = pd.concat([lm_row, summ])
print("Full results summary (deduplicated across all jobs):")
display(
    display_df.style
    .format({
        "mean_all":    "{:.3f}",
        "mean_pos":    "{:.3f}",
    }, na_rep="—")
    .background_gradient(subset=["iou>0 %"], cmap="RdYlGn")
    .background_gradient(subset=["mean_pos"], cmap="YlGn")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

exps_to_plot = TEXT_BASELINES + BOX_EXPS + MF_EXPS

# --- success rate ---
ax = axes[0]
vals = [summ.loc[e, "iou>0 %"] for e in exps_to_plot]
colors = [EXP_PALETTE.get(e, "gray") for e in exps_to_plot]
bars = ax.bar([EXP_LABELS[e] for e in exps_to_plot], vals, color=colors, edgecolor="white")
ax.axhline(100 * (all_df[all_df["experiment"]=="EXP-C"]["success"].mean()),
           ls="--", color=EXP_PALETTE["EXP-C"], lw=1.5, alpha=0.6, label="EXP-C baseline")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.5,
            f"{v:.1f}%", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("Success rate (IoU > 0) %")
ax.set_title("Localization success rate", fontweight="bold")
ax.tick_params(axis="x", labelsize=8)

# --- mean IoU when positive ---
ax = axes[1]
vals2 = [summ.loc[e, "mean_pos"] for e in exps_to_plot]
bars2 = ax.bar([EXP_LABELS[e] for e in exps_to_plot], vals2, color=colors, edgecolor="white")
ax.axhline(LM_EEC_MEAN_IOU, ls="--", color="black", lw=1.5, label=f"LM-EEC ({LM_EEC_MEAN_IOU})")
for b, v in zip(bars2, vals2):
    if not np.isnan(v):
        ax.text(b.get_x() + b.get_width()/2, v + 0.005,
                f"{v:.3f}", ha="center", fontsize=8, fontweight="bold")
ax.set_ylabel("Mean IoU (when IoU > 0)")
ax.set_title("Segmentation quality (positive pairs only)", fontweight="bold")
ax.legend()
ax.tick_params(axis="x", labelsize=8)

plt.suptitle("VLM-SAM3 Experiment Comparison — Jobs 465388 & 465389",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3  EXP-C-box Deep Dive: Bbox Parse Rate & IoU Distribution

The key question: when VLM outputs a bbox, does it produce a valid one? And when it does, what's the IoU?

In [ ]:
for exp in BOX_EXPS:
    g = all_df[all_df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    has_bbox = g["vlm_bbox"].notna() & (g["vlm_bbox"].astype(str) != "nan")
    not_vis  = (~has_bbox).sum()

    print(f"\n{'='*55}")
    print(f"  {exp}  ({len(g)} pairs)")
    print(f"{'='*55}")
    print(f"  VLM found bbox     : {has_bbox.sum()} ({100*has_bbox.mean():.1f}%)")
    print(f"  VLM said not_vis   : {not_vis} ({100*not_vis/max(1,len(g)):.1f}%)")
    print(f"  iou > 0            : {len(pos)}/{len(iou)} = {100*len(pos)/max(1,len(iou)):.1f}%")
    if len(pos):
        print(f"  mean IoU (pos)     : {pos.mean():.3f}   median: {pos.median():.3f}")
        print(f"  iou > 0.5          : {(pos>0.5).sum()} ({100*(pos>0.5).mean():.1f}%)")
        print(f"  iou > 0.75         : {(pos>0.75).sum()} ({100*(pos>0.75).mean():.1f}%)")
        print(f"  iou > 0.90         : {(pos>0.9).sum()} ({100*(pos>0.9).mean():.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, exp in zip(axes, BOX_EXPS):
    g = all_df[all_df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    not_vis = (~(g["vlm_bbox"].notna() & (g["vlm_bbox"].astype(str) != "nan"))).sum()

    # stacked bar: not_visible / failed_iou=0 / success
    iou_zero = (iou == 0).sum()
    ax.bar(["Pairs"], [not_vis],  color="#e74c3c", label="VLM: not visible")
    ax.bar(["Pairs"], [iou_zero], color="#f39c12",
           bottom=[not_vis], label="bbox found, IoU=0")
    ax.bar(["Pairs"], [len(pos)],  color="#2ecc71",
           bottom=[not_vis + iou_zero], label="IoU > 0")
    ax.set_title(f"{exp} — outcome breakdown\n(N={len(g)})", fontweight="bold")
    ax.set_ylabel("# pairs")
    ax.legend()

plt.suptitle("Bbox experiments: outcome breakdown", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# IoU distribution for positive pairs
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for ax, exp in zip(axes, BOX_EXPS):
    pos = all_df[all_df["experiment"] == exp]["iou"]
    pos = pos[pos > 0].dropna()
    ax.hist(pos, bins=30, color=EXP_PALETTE[exp], edgecolor="white")
    ax.axvline(LM_EEC_MEAN_IOU, color="black", ls="--", label=f"LM-EEC {LM_EEC_MEAN_IOU}")
    ax.axvline(pos.mean(), color="red", ls="-", label=f"mean={pos.mean():.3f}")
    ax.set_xlabel("IoU")
    ax.set_ylabel("Count")
    ax.set_title(f"{exp} — IoU distribution (positive pairs, n={len(pos)})", fontweight="bold")
    ax.legend()
plt.suptitle("IoU distribution — positive pairs only", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4  EXP-C-box vs Text Baselines

Two dimensions of comparison:
- **Localization rate** (iou>0%): how often does the approach find the object at all?
- **Segmentation quality** (mean IoU | success): when found, how precise is SAM3's mask?

In [ ]:
compare_exps = ["EXP-C", "EXP-C-cot", "EXP-C-box", "EXP-C-box-cot"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
metrics = [
    ("iou>0 %",   "Success / localization rate (%)",  "Localization rate"),
    ("mean_pos",  "Mean IoU (positive pairs)",         "Segmentation quality"),
    ("iou>0.5 %", "% pairs with IoU > 0.5",           "IoU > 0.5 rate"),
]

for ax, (col, ylabel, title) in zip(axes, metrics):
    vals   = [summ.loc[e, col] for e in compare_exps]
    colors = [EXP_PALETTE[e] for e in compare_exps]
    bars = ax.bar([EXP_LABELS[e] for e in compare_exps], vals, color=colors, edgecolor="white")
    for b, v in zip(bars, vals):
        if not np.isnan(float(v)):
            ax.text(b.get_x() + b.get_width()/2, float(v) + 0.3,
                    f"{float(v):.1f}", ha="center", fontsize=9, fontweight="bold")
    if col == "mean_pos":
        ax.axhline(LM_EEC_MEAN_IOU, ls="--", color="black", lw=1.5, label=f"LM-EEC ({LM_EEC_MEAN_IOU})")
        ax.legend()
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.tick_params(axis="x", labelsize=9)

handles = [mpatches.Patch(color=EXP_PALETTE[e], label=e) for e in compare_exps]
fig.legend(handles=handles, loc="upper right", bbox_to_anchor=(1.0, 1.0))
plt.suptitle("Bbox vs Text approaches — localization vs quality trade-off",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  EXP-C-box has HIGHER localization (49.4%) but LOWER quality (mean IoU=0.357)")
print("  EXP-C (text) has LOWER localization (31.7%) but HIGHER quality (mean IoU=0.686 > LM-EEC)")
print("  → The bottleneck is VLM localization, not SAM3 segmentation")

---
## 5  Per-Scenario Breakdown (EXP-C-box)

In [ ]:
SCENARIOS = {
    "basketball": ["basketball", "hoop", "Basket"],
    "cooking":    ["knife", "pan", "bowl", "kettle", "plate", "board", "noodle",
                   "egg", "paper", "fork", "spoon", "mug", "strainer", "spatula", "spaghetti"],
    "health":     ["CPR", "dummy", "covid", "swab", "test cassette", "test device",
                   "antigen", "extraction", "solution tube", "test kit", "instruction manual"],
    "music":      ["piano", "sheet", "guitar"],
    "bike":       ["bike", "bicycle", "wrench", "socket", "pump", "tube",
                   "handlebar", "grip", "stand", "tire", "trolley"],
}

box_df = all_df[all_df["experiment"] == "EXP-C-box"].copy()

scenario_stats = []
for scenario, keywords in SCENARIOS.items():
    pattern = "|".join(keywords)
    mask = box_df["object_name"].str.contains(pattern, case=False, na=False)
    g = box_df[mask]
    if len(g) == 0:
        continue
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    scenario_stats.append({
        "scenario":   scenario,
        "n":          len(g),
        "iou>0 %":    100 * len(pos) / max(1, len(iou)),
        "mean_pos":   pos.mean() if len(pos) else float("nan"),
        "iou>0.5 %": 100 * (pos > 0.5).mean() if len(pos) else 0,
    })

scen_df = pd.DataFrame(scenario_stats).set_index("scenario")
print("EXP-C-box — per-scenario breakdown:")
display(scen_df.style
    .format({"iou>0 %": "{:.1f}%", "mean_pos": "{:.3f}", "iou>0.5 %": "{:.1f}%"})
    .background_gradient(subset=["iou>0 %"], cmap="RdYlGn")
)

In [ ]:
SCENARIO_COLORS = {
    "basketball": "#e74c3c",
    "cooking":    "#f39c12",
    "health":     "#3498db",
    "music":      "#9b59b6",
    "bike":       "#2ecc71",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, ylabel, title in [
    (axes[0], "iou>0 %",  "Success rate (%)",      "Localization success rate by scenario"),
    (axes[1], "mean_pos", "Mean IoU (positive)",   "Segmentation quality by scenario"),
]:
    vals   = scen_df[col]
    colors = [SCENARIO_COLORS.get(s, "gray") for s in vals.index]
    bars   = ax.bar(vals.index, vals.values, color=colors, edgecolor="white")
    for b, v in zip(bars, vals.values):
        if not np.isnan(float(v)):
            ax.text(b.get_x() + b.get_width()/2, float(v) + 0.5,
                    f"{float(v):.1f}", ha="center", fontsize=10, fontweight="bold")
    if col == "mean_pos":
        ax.axhline(LM_EEC_MEAN_IOU, ls="--", color="black", lw=1.5,
                   label=f"LM-EEC ({LM_EEC_MEAN_IOU})")
        ax.legend()
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")

plt.suptitle("EXP-C-box — per-scenario analysis", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 6  Multiframe Ablation (EXP-F)

How many source frames should we show the VLM? Ablation: 1, 3, 5, 10 frames.

All EXP-F experiments use the instruct model (235b-instruct) with text output.

In [ ]:
mf_stats = []
for exp in MF_EXPS:
    g = all_df[all_df["experiment"] == exp].copy()
    iou = g["iou"].dropna()
    pos = iou[iou > 0]
    miss = (g["vlm_output"].isna() | (g["vlm_output"].astype(str).str.strip() == "")).sum()
    mf_stats.append({
        "experiment":   exp,
        "n_frames":     int(exp.replace("EXP-F-mf", "")),
        "N":            len(iou),
        "iou>0 %":      100 * len(pos) / max(1, len(iou)),
        "mean_pos":     pos.mean() if len(pos) else float("nan"),
        "iou>0.5 %":   100 * (pos > 0.5).mean() if len(pos) else 0,
        "iou>0.75 %":  100 * (pos > 0.75).mean() if len(pos) else 0,
        "missing_vlm": miss,
    })

mf_df = pd.DataFrame(mf_stats).set_index("experiment")
print("Multiframe ablation results:")
display(mf_df.style
    .format({"iou>0 %": "{:.1f}%", "mean_pos": "{:.3f}",
             "iou>0.5 %": "{:.1f}%", "iou>0.75 %": "{:.1f}%"})
    .background_gradient(subset=["iou>0 %"], cmap="RdYlGn")
    .background_gradient(subset=["mean_pos"], cmap="YlGn")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = mf_df["n_frames"]

# Success rate vs n_frames
ax = axes[0]
ax.plot(x, mf_df["iou>0 %"], "o-", color="#e74c3c", lw=2, ms=8, label="iou>0 %")
ax.set_xlabel("Number of source frames")
ax.set_ylabel("Success rate (IoU > 0) %")
ax.set_title("Localization success rate vs # frames", fontweight="bold")
ax.set_xticks([1, 3, 5, 10])
for xi, yi in zip(x, mf_df["iou>0 %"]):
    ax.annotate(f"{yi:.1f}%", (xi, yi), textcoords="offset points",
                xytext=(5, 5), fontsize=10, fontweight="bold")

# Quality vs n_frames
ax = axes[1]
ax.plot(x, mf_df["mean_pos"], "s-", color="#2ecc71", lw=2, ms=8, label="mean IoU | success")
ax.plot(x, mf_df["iou>0.5 %"] / 100, "^--", color="#3498db", lw=1.5, ms=6, label="iou>0.5 rate")
ax.axhline(LM_EEC_MEAN_IOU, ls="--", color="black", lw=1.5, label=f"LM-EEC ({LM_EEC_MEAN_IOU})")
ax.set_xlabel("Number of source frames")
ax.set_ylabel("IoU")
ax.set_title("Segmentation quality vs # frames", fontweight="bold")
ax.set_xticks([1, 3, 5, 10])
ax.legend()
for xi, yi in zip(x, mf_df["mean_pos"]):
    ax.annotate(f"{yi:.3f}", (xi, yi), textcoords="offset points",
                xytext=(5, -12), fontsize=10, fontweight="bold")

plt.suptitle("Multiframe ablation — EXP-F-mf{1,3,5,10}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey observation:")
print("  Fewer frames → better localization rate (mf1=33.1% >> mf5=17.6%)")
print("  3 frames → best quality when found (mean_pos=0.620)")
print("  More context images appear to confuse the VLM's localization")

In [ ]:
mf_data = all_df[all_df["experiment"].isin(MF_EXPS)].copy()
mf_data = mf_data[mf_data["iou"] > 0]  # positive pairs only

fig, ax = plt.subplots(figsize=(10, 5))
sns.violinplot(
    data=mf_data, x="experiment", y="iou",
    order=MF_EXPS,
    palette={e: EXP_PALETTE[e] for e in MF_EXPS},
    inner="quartile", cut=0, ax=ax
)
ax.axhline(LM_EEC_MEAN_IOU, ls="--", color="black", lw=1.5, label=f"LM-EEC ({LM_EEC_MEAN_IOU})")
ax.set_xlabel("")
ax.set_xticklabels([EXP_LABELS[e] for e in MF_EXPS])
ax.set_ylabel("IoU")
ax.set_title("IoU distribution — positive pairs — multiframe ablation", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7  Best & Worst Examples (EXP-C-box)

In [ ]:
box_all = all_df[all_df["experiment"] == "EXP-C-box"].copy()

print("TOP-10 IoU pairs (EXP-C-box):")
top10 = box_all.nlargest(10, "iou")[
    ["object_name", "take_uid", "frame", "vlm_bbox", "iou"]
]
for _, r in top10.iterrows():
    print(f"  [{r['object_name']:30s}]  bbox={str(r['vlm_bbox'])[:30]}  iou={float(r['iou']):.3f}")

print("\nBOTTOM-10 (IoU>0, EXP-C-box):")
pos = box_all[box_all["iou"] > 0]
bot10 = pos.nsmallest(10, "iou")[["object_name", "take_uid", "frame", "vlm_bbox", "iou"]]
for _, r in bot10.iterrows():
    print(f"  [{r['object_name']:30s}]  bbox={str(r['vlm_bbox'])[:30]}  iou={float(r['iou']):.4f}")

In [ ]:
# IoU bucket analysis
box_pos = box_all[box_all["iou"] > 0]["iou"].dropna()

buckets = {
    "0 < IoU ≤ 0.1":  ((box_pos > 0) & (box_pos <= 0.1)).sum(),
    "0.1 < IoU ≤ 0.25": ((box_pos > 0.1) & (box_pos <= 0.25)).sum(),
    "0.25 < IoU ≤ 0.5":  ((box_pos > 0.25) & (box_pos <= 0.5)).sum(),
    "0.5 < IoU ≤ 0.75":  ((box_pos > 0.5) & (box_pos <= 0.75)).sum(),
    "0.75 < IoU ≤ 1.0":  (box_pos > 0.75).sum(),
}

fig, ax = plt.subplots(figsize=(9, 4))
colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#27ae60"]
bars = ax.bar(buckets.keys(), buckets.values(), color=colors, edgecolor="white")
for b, v in zip(bars, buckets.values()):
    ax.text(b.get_x() + b.get_width()/2, v + 1,
            str(v), ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("# pairs")
ax.set_title("EXP-C-box — IoU bucket distribution (positive pairs only)", fontweight="bold")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

total_pos = len(box_pos)
print(f"Total positive pairs: {total_pos}")
for k, v in buckets.items():
    print(f"  {k:25s}: {v:4d} ({100*v/total_pos:.1f}%)")

---
## 8  Key Findings & Conclusions

In [ ]:
box = all_df[all_df["experiment"] == "EXP-C-box"]
expc = all_df[all_df["experiment"] == "EXP-C"]
expc_cot = all_df[all_df["experiment"] == "EXP-C-cot"]
mf3 = all_df[all_df["experiment"] == "EXP-F-mf3"]

def pct_success(df): return 100 * (df["iou"] > 0).mean()
def mean_pos(df):
    pos = df[df["iou"] > 0]["iou"]
    return pos.mean()

print("="*70)
print("  KEY FINDINGS — Jobs 465388 & 465389")
print("="*70)

print("\n  LOCALIZATION RATE (iou > 0):")
print(f"    LM-EEC baseline:      ~100% (traditional method, always produces a mask)")
print(f"    EXP-C-box  (bbox):     {pct_success(box):.1f}%  ← HIGHEST across all VLM experiments")
print(f"    EXP-C-cot  (text):     {pct_success(expc_cot):.1f}%")
print(f"    EXP-C      (text):     {pct_success(expc):.1f}%")
print(f"    EXP-F-mf3  (3frame):   {pct_success(mf3):.1f}%")

print("\n  SEGMENTATION QUALITY (mean IoU | success):")
print(f"    LM-EEC baseline:       0.660  ← reference")
print(f"    EXP-C      (text):     {mean_pos(expc):.3f}  ← BEATS LM-EEC!")
print(f"    EXP-C-cot  (text):     {mean_pos(expc_cot):.3f}")
print(f"    EXP-F-mf3  (3frame):   {mean_pos(mf3):.3f}")
print(f"    EXP-C-box  (bbox):     {mean_pos(box):.3f}  ← lower (bbox less precise than text)")

print("\n  MAIN CONCLUSIONS:")
conclusions = [
    "1. BOTTLENECK is VLM localization, not SAM3 segmentation.",
    "   When VLM finds the object, SAM3 segments it well (IoU=0.686 > LM-EEC=0.660).",
    "",
    "2. BBOX approach finds objects more often (49.4%) but SAM3 box prompts",
    "   are less precise than text prompts (mean IoU=0.357 vs 0.686).",
    "",
    "3. CoT HURTS bbox (49.4% → 30.4%): reasoning causes coordinate second-guessing.",
    "",
    "4. MULTIFRAME: 1 frame = best localization, 3 frames = best quality when found.",
    "   More images confuse the VLM — more is not better here.",
    "",
    "5. SCENARIO DIFFICULTY: Music (piano/guitar) easiest (96.9%),",
    "   Cooking hardest (44.7%), small bike parts worst quality (mean IoU=0.247).",
    "",
    "6. NEXT STEP: Fix VLM localization. Possible directions:",
    "   - Hybrid: bbox → crop → re-prompt SAM3 with text on cropped region",
    "   - Better prompts / few-shot examples",
    "   - Qualitative error analysis: what does the VLM say when it fails?",
]
for c in conclusions:
    print(f"  {c}")

print("\n" + "="*70)